# Step 2 — Training ACT on Your Lebai Demos

This notebook trains an **ACT** policy on the LeRobot dataset produced by `convert_alllog_to_lerobot.ipynb`. At the end, you'll have a checkpoint that the inference notebook can use to drive the robot.

## What you'll learn

1. **The mental model behind ACT.** Why we predict *chunks* of future actions rather than one action at a time, and why there's a VAE bolted onto the encoder.
2. **The shape of a LeRobot dataset.** Features, episodes, frames, dataloaders.
3. **Hyperparameters that matter most** for a 6-DOF arm + gripper at 10 Hz — most ACT defaults assume bimanual ALOHA at 50 Hz, so a few things need adjusting.
4. **How to monitor training** and decide when to stop.

## Prerequisites

- A LeRobot dataset built by the converter notebook. Default location: `$HF_LEROBOT_HOME/<REPO_ID>/`.
- A CUDA GPU with ≥8 GB VRAM (12 GB comfortable).
- `lerobot` installed.

## Action / state shape

Your dataset has:
- **state**: `(7,)` = `[jp0..jp5, claw_amplitude]` — current joint positions + current gripper opening
- **action**: `(7,)` = `[tgt_jp0..tgt_jp5, next_claw_amplitude]` — commanded joint targets + commanded gripper opening

The model auto-derives input/output shapes from the dataset, so you don't hardcode 7 anywhere — but it's worth knowing what each dimension means when debugging.


## What is ACT, in one screen

The original ACT paper (Zhao et al., *Learning Fine-Grained Bimanual Manipulation with Low-Cost Hardware*, RSS 2023) made two architectural choices that account for almost all of its performance:

**1. Predict a chunk of future actions, not just the next one.**

Naïve behavior cloning predicts $a_t$ from $o_t$. The problem: small prediction errors push the robot to states it never saw during training, where it makes bigger errors, and so on. This is *compounding error*. ACT instead predicts the next $K$ actions all at once. Then it executes some prefix of that chunk before predicting again. This shortens how often errors get a chance to compound.

At inference, ACT also does **temporal ensembling**: at every timestep it averages the actions predicted by recent overlapping chunks, producing a smooth trajectory. We'll see this in the inference notebook.

**2. CVAE encoder for handling demo variability.**

Human demonstrations of the same task differ in style, speed, and exact trajectory. If you train a deterministic network to predict the average of two valid demos, you can get a "mean trajectory" that does neither. ACT's solution is a **conditional VAE**: during training, an encoder maps the demo's action sequence to a latent style vector $z$; the decoder predicts actions conditioned on $(o_t, z)$. At inference, you set $z = 0$ (the prior mean) and let the policy do "average style" — which works, because the decoder isn't asked to model the variability itself.

The architecture is otherwise a standard Transformer encoder–decoder. Vision backbone is a per-image ResNet, states are projected to tokens, the decoder cross-attends to encoder outputs.


## 1. Setup

In [ ]:
# !pip install lerobot


In [ ]:
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch.utils.data import DataLoader

from lerobot.common.datasets.lerobot_dataset import LeRobotDataset
from lerobot.common.policies.act.modeling_act import ACTPolicy
from lerobot.common.policies.act.configuration_act import ACTConfig

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


## 2. Load the dataset

`DATASET_REPO_ID` should match what you set in the converter notebook.


In [ ]:
DATASET_REPO_ID = "local/lebai_duck_pick"  # must match REPO_ID from the converter

dataset_meta = LeRobotDataset(DATASET_REPO_ID).meta
print(f"  episodes: {dataset_meta.total_episodes}")
print(f"  frames:   {dataset_meta.total_frames}")
print(f"  fps:      {dataset_meta.fps}")
print(f"  tasks:    {len(dataset_meta.tasks)}")
print()
print("Features:")
for name, spec in dataset_meta.features.items():
    print(f"  {name}: dtype={spec['dtype']}, shape={spec.get('shape')}")


## 3. Visualize before training

The single highest-ROI sanity check in the entire pipeline. Bugs in the converter (wrong color channel, action sign flipped, swapped joints) show up here as "the picture looks wrong" or "joint 3 doesn't move at all" — and they are nearly invisible from training loss alone.


In [ ]:
ds_inspect = LeRobotDataset(DATASET_REPO_ID)
sample = ds_inspect[0]
image_keys = [k for k in sample.keys() if k.startswith("observation.images.")]
print("Image keys:", image_keys)

fig, axes = plt.subplots(1, len(image_keys), figsize=(5 * len(image_keys), 5))
if len(image_keys) == 1:
    axes = [axes]
for ax, key in zip(axes, image_keys):
    img = sample[key].permute(1, 2, 0).cpu().numpy()
    ax.imshow(img); ax.set_title(key.replace("observation.images.", "")); ax.axis("off")
plt.tight_layout(); plt.show()

print(f"state:  {sample['observation.state'].numpy().round(3)}")
print(f"action: {sample['action'].numpy().round(3)}")
print(f"task:   {sample['task']!r}")


In [ ]:
# Plot one episode's state and action trajectories side by side.
ep0_from = ds_inspect.episode_data_index["from"][0].item()
ep0_to   = ds_inspect.episode_data_index["to"][0].item()
frames = [ds_inspect[i] for i in range(ep0_from, ep0_to)]

states  = torch.stack([f["observation.state"] for f in frames]).numpy()
actions = torch.stack([f["action"] for f in frames]).numpy()
fps = dataset_meta.fps
t = np.arange(len(states)) / fps

# Last dim is gripper; plot it on its own axis since the scale is different.
state_dim = states.shape[1]
arm_dims = state_dim - 1   # assumes last dim is gripper

fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharex=True)
for i in range(arm_dims):
    axes[0, 0].plot(t, states[:, i], label=f"j{i}")
    axes[0, 1].plot(t, actions[:, i], label=f"j{i}")
axes[1, 0].plot(t, states[:, -1], color="black", label="gripper")
axes[1, 1].plot(t, actions[:, -1], color="black", label="gripper")
axes[0, 0].set_title("state — arm joints"); axes[0, 0].legend(loc="upper right")
axes[0, 1].set_title("action — arm joints"); axes[0, 1].legend(loc="upper right")
axes[1, 0].set_title("state — gripper amplitude"); axes[1, 0].set_xlabel("time (s)")
axes[1, 1].set_title("action — gripper amplitude"); axes[1, 1].set_xlabel("time (s)")
for a in axes.flat: a.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()


**What you want to see**

- **Joint trajectories smooth, no jagged jumps.** Jumps usually mean dropped frames or a frame-pairing bug.
- **Gripper makes discrete transitions** between roughly two values (open and closed). If it's noisy continuous wobble, your gripper logging is broken.
- **Action ≠ state.** They should be related but not identical. If they overlap exactly, your converter is treating state as action and the policy will learn a no-op.
- **Different episodes look different.** Plot ep1, ep2, ep3 too. If they're identical, your episode grouping is broken.


## 4. Configure ACT

`chunk_size` is the most important hyperparameter. ALOHA uses 100 actions at 50 Hz = 2 s of lookahead. **You're at 10 Hz**, so 2 s of lookahead = 20 chunks. I'd start with **`chunk_size=32`** (3.2 s) — slower 6-DOF tasks tend to benefit from a bit more lookahead.

`n_action_steps == chunk_size` for training. At inference, you'll execute fewer than that and let temporal ensembling smooth the rest.


In [ ]:
CHUNK_SIZE = 32
N_ACTION_STEPS = CHUNK_SIZE

delta_timestamps = {
    "action": [t / fps for t in range(CHUNK_SIZE)],
}
dataset = LeRobotDataset(DATASET_REPO_ID, delta_timestamps=delta_timestamps)
print(f"Training frames: {len(dataset)}  (each yields a chunk of {CHUNK_SIZE} actions)")


In [ ]:
cfg = ACTConfig(
    n_obs_steps=1,
    chunk_size=CHUNK_SIZE,
    n_action_steps=N_ACTION_STEPS,

    vision_backbone="resnet18",
    pretrained_backbone_weights="ResNet18_Weights.IMAGENET1K_V1",
    replace_final_stride_with_dilation=False,

    dim_model=512,
    n_heads=8,
    dim_feedforward=3200,
    n_encoder_layers=4,
    n_decoder_layers=1,   # original ACT impl uses 1, not 7 as in the paper

    use_vae=True,
    latent_dim=32,
    n_vae_encoder_layers=4,
    kl_weight=10.0,

    dropout=0.1,
)

cfg.input_features = {k: v for k, v in dataset.features.items() if k.startswith("observation.")}
cfg.output_features = {"action": dataset.features["action"]}

policy = ACTPolicy(cfg, dataset_stats=dataset.meta.stats)
policy.to(device)
policy.train()

n_params = sum(p.numel() for p in policy.parameters())
print(f"Policy: {n_params/1e6:.1f}M parameters")
print(f"  predicting actions of shape {dataset.features['action']['shape']}")


## 5. Training loop

Standard supervised setup: AdamW, cosine schedule, MSE on actions plus a KL term from the CVAE. `policy.forward(batch)` returns the combined loss.

- **Steps**: 50k is a sensible default for a small dataset. Watch the loss curve; if still descending at 50k, extend.
- **Batch size**: 8 fits in 12 GB. 16 or 32 on bigger cards.
- **LR**: 1e-4 standard. LeRobot internally gives the vision backbone a lower LR (1e-5) to preserve pretrained features.


In [ ]:
BATCH_SIZE = 8
NUM_STEPS = 50_000
LOG_EVERY = 200
SAVE_EVERY = 5_000

CHECKPOINT_DIR = Path("./checkpoints/act_run01")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

dataloader = DataLoader(
    dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=4, pin_memory=device.type == "cuda", drop_last=True,
)
def cycle(loader):
    while True:
        for batch in loader:
            yield batch
step_iter = cycle(dataloader)

optimizer = torch.optim.AdamW(policy.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_STEPS)

print(f"Will run {NUM_STEPS} steps  bs={BATCH_SIZE}  -> {CHECKPOINT_DIR.resolve()}")


In [ ]:
loss_history = []
t0 = time.time()

for step in range(NUM_STEPS):
    batch = next(step_iter)
    batch = {k: v.to(device, non_blocking=True) if torch.is_tensor(v) else v
             for k, v in batch.items()}

    output_dict = policy.forward(batch)
    loss = output_dict["loss"]

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(policy.parameters(), max_norm=10.0)
    optimizer.step()
    scheduler.step()

    loss_history.append(loss.item())

    if step % LOG_EVERY == 0:
        elapsed = time.time() - t0
        rate = (step + 1) / elapsed
        print(f"step {step:6d}  loss={loss.item():.4f}  lr={scheduler.get_last_lr()[0]:.2e}  ({rate:.1f} step/s)")

    if (step + 1) % SAVE_EVERY == 0:
        ckpt_path = CHECKPOINT_DIR / f"step_{step+1:06d}"
        policy.save_pretrained(ckpt_path)
        print(f"  -> saved {ckpt_path}")

final_ckpt = CHECKPOINT_DIR / "final"
policy.save_pretrained(final_ckpt)
print(f"\nDone. Final: {final_ckpt}")


## 6. Loss curve

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(loss_history, alpha=0.4, label="raw")
window = max(1, len(loss_history) // 100)
if len(loss_history) >= window:
    smoothed = np.convolve(loss_history, np.ones(window)/window, mode="valid")
    ax.plot(np.arange(window-1, len(loss_history)), smoothed, label=f"smoothed (w={window})")
ax.set_xlabel("step"); ax.set_ylabel("loss"); ax.legend(); ax.grid(True, alpha=0.3)
ax.set_title("ACT training loss"); plt.tight_layout(); plt.show()


## 7. Next steps

You now have `./checkpoints/act_run01/final/`. Open **`act_inference.ipynb`** to load and run it.

**If loss stays high:**
- Re-inspect more frames. Visualize episodes 1–10 of state and action.
- Check action representation. The state vs action plot above should differ.
- Increase `chunk_size`, increase steps, or collect more demos.

**When to switch to Diffusion Policy:**
If your demos clearly have multiple equally-valid strategies (grasp the bottle from the left or the right) and trained ACT does a wishy-washy averaged motion that fails — that's the multi-modality problem ACT handles poorly. Diffusion Policy handles it. Same dataset, drop-in replacement of the policy class.
